<a href="https://colab.research.google.com/github/OvaltineSamuel/MLSN_F26_Group2/blob/main/CNN_Train.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# generic setup cell we can re-use for all notebooks
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import os, sys
ROOT = '/content/drive/MyDrive/Google_Colab_Share_Folder/MLSN_F26_Group_2_(CRISPR)'
DATA = os.path.join(ROOT, 'data', 'processed')
MODELS = os.path.join(ROOT, 'models')
sys.path.append(os.path.join(ROOT, 'src'))

print(os.path.exists(ROOT))  # should print True

Mounted at /content/drive
True


## 1. Load The Dataset

Load in the one-hot tensor dataset for training in PyTorch from ../data/processed


In [ ]:
import pandas as pd
Deephf_all_df = pd.read_csv(os.path.join(DATA, 'deephf_cleaned_all_59852.csv'))
Deephf_all_df.head()

,gRNA_ID,gRNA_Seq,PAM,21mer,WT_Efficiency,eSpCas9_Efficiency,SpCas9_HF1_Efficiency
0,GRNA-1,AAAAAAAAACTCCAAAACCC,TGG,AAAAAAAAACTCCAAAACCCT,0.168570,0.142063,0.093147
1,GRNA-2,AAAAAACAACAAGAAGCACA,AGG,AAAAAACAACAAGAAGCACAA,0.099624,0.051901,0.064951
2,GRNA-3,AAAAAACACAAGCAAGACCG,TGG,AAAAAACACAAGCAAGACCGT,0.247750,0.043573,0.061797
3,GRNA-4,AAAAAACAGATGCCACCTGT,GGG,AAAAAACAGATGCCACCTGTG,0.141232,0.052343,0.057246
4,GRNA-5,AAAAAACCCGTAGATAGCCT,CGG,AAAAAACCCGTAGATAGCCTC,0.045803,0.046234,0.067596


## 2. Define the Bio features
Implement all biological features as functions:


In [ ]:

import Bio.SeqUtils
from Bio.SeqUtils import MeltingTemp as mt
import RNA
import numpy as np




SEQ_LEN = 21 #20-nt spacer plus first nt in PAM
SPACER_LEN = 20

SCAFFOLD = "GTTTTAGAGCTAGAAATAGCAAGTTAAAATAAGGCTAGTCCGTTATCAACTTGAAAAAGTGGCACCGAGTCGGTGCTTTTT"

TARGET_COLUMNS = ["wt_efficiency", "esp_efficiency", "hf_efficiency"]

SCAFFOLD_REF = RNA.fold(SCAFFOLD.replace("T", "U"))[0]

BASES = "ACGT"
DINUCS = [a + b for a in BASES for b in BASES]

GLOBAL_FEATURE_NAMES = (
    ["gc_content", "gc_count"]
    + [f"count_{b}" for b in BASES]
    + [f"count_{d}" for d in DINUCS]
    + ["tm_full", "tm_1_4", "tm_16_20", "tm_5_13",
       "mfe_spacer", "mfe_full", "mfe_difference",
       "ensemble_energy", "ensemble_diversity",
       "scaffold_integrity", "scaffold_bp_lost",
       "accessibility_mean", "accessibility_pam_prox",
       "intra_pairing_mean", "scaffold_pairing_mean"]
)
POSITIONAL_CHANNEL_NAMES = ["p_unpaired", "p_intra", "p_scaffold"]
# p_intra is hairpin within spacer, scaffold is within scaffold

def melting_temp(seq:str)-> float:
  return float(mt.Tm_NN(seq))

def fold(rna:str) -> tuple[str,float]:
  structure, mfe = RNA.fold(rna)
  return structure, mfe

def unpaired_probabilities(rna:str, spacer_len: int=SPACER_LEN) -> np.ndarray:
  """Per-position probability that this NT
  is unpaired. Used to encode the position
  accessibility: ie, what are the odds a spacer
  NT is free to bind with target DNA

  Returns (p_unpaired, p_intra, p_scaffold), each length spacer_len."""

  fc = RNA.fold_compound(rna)
  # ViennaRNA object holding sequence and
  # energy rules

  _, mfe = fc.mfe()
  # finds most stable fold and saves the free energy of it

  fc.exp_params_rescale(mfe)
  #to avoid floating point overflow when we partition

  fc.pf()
  #partition function, weighs all folds by stability

  bpp = fc.bpp()
  #base pair probability matrix:
  #bpp[i][j] is probability position i is
  # paired with position j

  n = len(rna)
  intra = np.zeroes(spacer_len, dtype=np.float64)
  scaffold = np.zeros(spacer_len, dtype=np.float64)
  for i in range(1, n+1):
    for j in range(i+1, n+1):
      p = bpp[i][j]
      for a, b in ((i, j), (j, i)):
                if a <= spacer_len:
                    if b <= spacer_len:
                        intra[a - 1] += p
                    else:
                        scaf[a - 1] += p
  unpaired = np.clip(1.0 - intra - scaf, 0.0, 1.0)
  # loops over every (i,j), if probability is nonzero
  # adds the probability of those 2 being paired such
  # that paired[i] = P(i being paired with anything)

  return npaired, np.clip(intra, 0, 1), np.clip(scaf, 0, 1)

def _scaffold_integrity(full_structure: str, spacer_len: int = SPACER_LEN) -> tuple[float, float]:
    ref = SCAFFOLD_REF
    tail = full_structure[spacer_len:]
    kept = sum(1 for a, b in zip(tail, ref) if a == b)
    return kept / len(ref), float(len(ref) - kept)

def compute_bio_features(seq21:str) -> np.ndarray:
   """Returns (global_features (F,), positional_channels (3, 21))."""
  seq21 = seq21.strip().upper().replace("U","T")
  spacer = seq21[:20]

  gc_count = sum(1 for b in spacer if b in "GC")
  mono = [float(spacer.count(b)) for b in BASES]
  di = [float(sum(1 for i in range(len(spacer)-1) if spacer[i:i+2]==d)) for d in DINUCS]


  tm_full = melting_temp(spacer)
  tm_pam_distal_5 = melting_temp(spacer[0:5])
  tm_mid_8 = melting_temp(spacer[5:13])
  tm_pam_proximal_5 = melting_temp(spacer[15:20])

  spacer_rna = spacer.replace("T", "U")
  full_rna = (spacer + SCAFFOLD).replace("T", "U")

  _, mfe_spacer = fold(spacer_rna)
  struct_full, mfe_full = fold(full_rna)

  fc = RNA.fold_compound(full_rna)
  _, m = fc.mfe()
  fc.exp_params_rescale(m)
  ens_energy = float(fc.pf()[1])
  ens_div = float(fc.mean_bp_distance())
  # fold diversity: high means many folds with reasonable likelihood

  unpaired, intra, scaf = unpaired_probabilities(full_rna)

  integrity, bp_lost = _scaffold_integrity(struct_full)

  glob = np.array([
      gc_count/SPACER_LEN,
      float(gc_count),
  ] + mono + di + [
      tm_full, tm_pam_distal_5, tm_mid_8, tm_pam_proximal_5,
      float(mfe_spacer), float(mfe_full),
      ens_energy, ens_div,
      integrity, bp_lost,
      float(unpaired.mean()),
      float(unpaired[15:20].mean()),
      float(intra.mean()),
      float(scaf.mean())
  ] dtype = np.float32)

  chans = np.zeros((3, SEQ_LEN), dtype=np.float32)
  chans[0,:SPACER_LEN] = unpaired
  chans[1,:SPACER_LEN] = intra
  chans[2,:SPACER_LEN] = scaf

  return glob, chans


  return np.array([
      gc_content, float(gc_count),
      tm_full, tm_pam_proximal_5, tm_mid_8, tm_pam_distal_5,
      mfe_spacer, mfe_full, mfe_full - mfe_spacer,
      float(unpaired.mean()),
      float(unpaired[15:20].mean()),
  ], dtype=np.float32)

def bio_matrix(seqs: list[str], cache_path: str| None = None) -> np.ndarray:
  """ (N, 11), cache the biofeature results"""
  if cache_path:
    try:
      cached = np.load(cache_path)
      if len(cached) == len(seqs):
        print(f"loaded cached bio features from {cache_path}")
        return cached
    except FileNotFoundError:
      pass

    rows = np.zeros((len(seqs), 11), dtype=np.float32)
    for i, s in enumerate(seqs):
      rows[i] = compute_bio_features(s)
      if (i+1) % 2000 == 0:
        print(f" bio features: {i+1}/{len(seqs)}")

    if cache_path:
      np.save(cache_path, rows)
    return rows
class Standardizer:
  """biofeatures are on different scales"""

  def __init__(self):
    self.mean = None
    self.std = None

  def fit(self, x):
    self.mean = x.mean(axis=0)
    self.std = x.std(axis=0)
    return self

  def transform(self, x):
    return ((x - self.mean) / self.std).astype(np.float32)




## Build the Model
A multi-modal CNN which takes bio features after convolution

In [ ]:
from typing import dataclass_transform
import torch
import torch.nn as nn
import scipy

@dataclass
class ModelConfig:
  kernel_widths: tuple[int] = (3,5,7)
  channels_per_width: int=64
  bottleneck_channels: int = 32
  bio_embed_dim: int = 32
  hidden_dims: tuple[int] = (256, 128)
  dropout: float = 0.3
  n_bio_features: int = 11
  n_targets: int = len(TARGET_COLUMNS)
  use_bio: bool = True
  pam_proximal_skip: int = 4  # raw one-hot of the last N positions, 0 to disable

class GuideCNN(nn.Module):
  def __init__(self, cfg: ModelConfig) -> None:
    super().__init__()
    self.cfg = cfg

    self.branches = nn.ModuleList(
        [
            nn.Conv1d(4, cfg.channels_per_width, k, padding=k//2)
            for k in cfg.kernel_widths
        ]
    )
    trunk_channels = cfg.channels_per_width * len(cfg.kernel_widths)

    self.bn1 = nn.BatchNorm1d(trunk_channels)
# normalize after 3 paralel convolutions
    self.conv2 = nn.Conv1d(trunk_channels, trunk_channels, kernel_size=3, padding=1)
# convolve again with kernel width 3, searching for deeper patterns
    self.bn2 = nn.BatchNorm1d(trunk_channels)
# same normalize again, needs to be a seperate object because
# because each normalization error keeps its own stats

  self.bottleneck = nn.Conv1d(trunk_channels, cfg.bottleneck_channels, kernel_size=1)

  seq_dim = cfg.bottleneck_channels * SEQ_LEN

  skip_dim = 4*cfg.pam_proximal_skip
#===================BIO FEATURES===============================

# Bio has it's own MLP so 11 features aren't drowned out by the conv features

if cfg.use_bio:
  self.bio_mlp = nn.Sequential(
      nn.Linear(cfg.n_bio_features, cfg.bio_embed_dim),
      nn.ReLU(),
      nn.Linear(cfg.bio_embed_dim, cfg.bio_embed_dim),
      nn.ReLU(),
  )
  bio_dim = cfg.bio_embed_dim
else:
  self.bio_mlp = None
  bio_dim = 0

  # reduce the risk of drowning out features: we should see how much difference the bio
  # features make, if they need more influence

layers: list[nn.Module] = []
in_dim = seq_dim + skip_dim + bio_dim
for h in cfg.hidden_dims:
  layers += [nn.Linear(in_dim, h), nn.ReLU(), nn.Dropout(cfg.dropout)]
  # consider adding batch normalization after linear for stability
  in_dim = h

layers.append(nn.Linear(in_dim, cfg.n_targets))
#final output

self.head = nn.Sequential(*layers)

def forward(self, x: torch.Tensor, bio: torch.Tensor | None = None) -> torch.Tensor:
  # x: (N,4,21), b(x): (N,64,21)
  h = torch.cat([nn.functional.relu(b(x)) for b in self.branches], dim=1)
  # h: (N,192,21), the 3 outputs stack along the channel axis

  h = self.bn1(h)
  # per channel normalization

  h = nn.functional.relu(self.bn2(self.conv2(h)))
  # another width-3 convolve, then normalize then zero negatives

  h = nn.functional.relu(self.bottleneck(h))

  parts = [h.flatten(1)]

  if self.cfg.pam_proximal_skip:
    #bypass convolutions for the pam proximal end
    parts.append(x[:,:,-self.cfg.pam_proximal_skip :].flatten(1))

    if self.bio_mlp is not None:
      if bio is None:
        raise ValueError("model was built with use_bio=True but bio is None")
      parts.append(self.bio_mlp(bio))

    # since efficiency is a fraction in [0,1]

    return torch.sigmoid(self.head(torch.cat(parts, dim=1)))










## 3. Training

In [ ]:
def spearman_per_target(y_true: np.ndarray, y_pred: np.ndarray) -> list[float]:
  return[
      float(spearmanr(y_true[:,i], y_pred[:,i]).statistic)
      for i in range(y_true.shape[1])
  ]

@torch.no_grad()
def evaluate(model: nn.Module, loader: torch.utils.data.DataLoader, device: str) -> tuple[float, list[float]]:
  model.eval()
  preds, trues, total_loss, n = [], [], 0.0, 0
  for xb, bb, yb in loader:
    xb, bb, yb = xb.to(device), bb.to(device), yb.to(device)
    # run on GPUs

    out = model(xb, bb if model.cfg.use_bio else None)

    n += yb.numel()

    preds.append(out.cpu().numpy())
    trues.append(yb.cpu().numpy())

  y_pred, y_true = np.concatenate(preds), np.concatenate(trues)

  return total_loss / n, spearman_per_target(y_true, y_pred)

def train(
    model: nn.Module,
    train_loader: torch.utils.data.DataLoader,
    val_loader: torch.utils.data.DataLoader,
    *,
    epochs: int = 60,
    lr: float = 1e-3,
    weight_decay: float = 1e-4,
    patience: int = 10,
    device: str = "cpu",
    checkpoint: str = "best_model.pt",
) -> nn.Module:
  model.to(device)
  opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)

  sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, mode="max", factor=0.5, patience=4)



